# Heatmap Data Exploration

Investigating `heatmap_past_data.csv` and `heatmap_future_predictions.csv`.

Key questions:
1. Is `is_a_prediction` uniform across each file?
2. Why is `heatmap_past_data` ~2× the size of `heatmap_sorted`?
3. Are there duplicate `leto_mesec` entries per segment?

> **Note:** The unique identifier for a road segment is the pair `(odsek_id, ggo)`, not just `odsek_id`.

In [2]:
import pandas as pd

past   = pd.read_csv("../data/heatmap_past_data.csv")
future = pd.read_csv("../data/heatmap_future_predictions.csv")
sorted_df = pd.read_csv("../data/heatmap_sorted.csv")

print("Shapes:")
print(f"  heatmap_past_data:          {len(past):>10,} rows  {past.shape[1]} cols")
print(f"  heatmap_future_predictions: {len(future):>10,} rows  {future.shape[1]} cols")
print(f"  heatmap_sorted:             {len(sorted_df):>10,} rows  {sorted_df.shape[1]} cols")
print(f"\nPast / Sorted ratio: {len(past)/len(sorted_df):.2f}x")

print("\nColumns:")
for name, df in [("past", past), ("future", future), ("sorted", sorted_df)]:
    print(f"  {name:8s}: {list(df.columns)}")

Shapes:
  heatmap_past_data:           8,225,663 rows  5 cols
  heatmap_future_predictions:    413,004 rows  5 cols
  heatmap_sorted:              4,109,455 rows  4 cols

Past / Sorted ratio: 2.00x

Columns:
  past    : ['ggo', 'odsek_id', 'leto_mesec', 'target', 'is_a_prediction']
  future  : ['ggo', 'odsek_id', 'leto_mesec', 'target', 'is_a_prediction']
  sorted  : ['ggo', 'odsek_id', 'leto_mesec', 'target']


## 1. `is_a_prediction` — is the value uniform within each file?

In [3]:
for name, df in [("heatmap_past_data", past), ("heatmap_future_predictions", future)]:
    vals = df["is_a_prediction"].value_counts()
    unique = df["is_a_prediction"].unique()
    is_uniform = df["is_a_prediction"].nunique() == 1
    print(f"=== {name} ===")
    print(vals.to_string())
    print(f"Unique values: {unique}")
    print(f"Uniform (only one value)? {'YES' if is_uniform else 'NO — MIXED!'}")
    print()

=== heatmap_past_data ===
is_a_prediction
False    8225663
Unique values: [False]
Uniform (only one value)? YES

=== heatmap_future_predictions ===
is_a_prediction
True    413004
Unique values: [ True]
Uniform (only one value)? YES



## 2. Duplicate `leto_mesec` per segment — using `(odsek_id, ggo)` as unique key

In [4]:
SEGMENT_KEY = ["odsek_id", "ggo"]
ROW_KEY = ["odsek_id", "ggo", "leto_mesec"]

# --- past data ---
n_unique_segments_past = past[SEGMENT_KEY].drop_duplicates().shape[0]
n_unique_rows_past     = past[ROW_KEY].drop_duplicates().shape[0]

print("=== heatmap_past_data ===")
print(f"  Total rows:                    {len(past):>10,}")
print(f"  Unique (odsek_id, ggo):        {n_unique_segments_past:>10,}")
print(f"  Unique (odsek_id, ggo, month): {n_unique_rows_past:>10,}")
print(f"  Duplicate (segment+month) rows: {len(past) - n_unique_rows_past:>9,}")

# --- heatmap_sorted for reference ---
# sorted has no ggo column — use odsek_id alone there
n_sorted_pairs = sorted_df[["odsek_id","leto_mesec"]].drop_duplicates().shape[0]
print(f"\n=== heatmap_sorted (reference) ===")
print(f"  Total rows:                    {len(sorted_df):>10,}")
print(f"  Unique (odsek_id, month):      {n_sorted_pairs:>10,}")

=== heatmap_past_data ===
  Total rows:                     8,225,663
  Unique (odsek_id, ggo):            34,417
  Unique (odsek_id, ggo, month):  8,225,663
  Duplicate (segment+month) rows:         0

=== heatmap_sorted (reference) ===
  Total rows:                     4,109,455
  Unique (odsek_id, month):       3,363,814


In [5]:
# How many times does each (odsek_id, ggo, leto_mesec) appear?
counts = past.groupby(ROW_KEY).size().value_counts().sort_index()
print("Distribution of rows per (odsek_id, ggo, leto_mesec):")
print(counts.rename("num_triplets").to_string())

Distribution of rows per (odsek_id, ggo, leto_mesec):
1    8225663


In [6]:
# Show a sample of duplicated triplets to understand what differs between them
dup_mask = past.duplicated(subset=ROW_KEY, keep=False)
dups = past[dup_mask].sort_values(ROW_KEY)
print(f"Rows involved in duplicates: {dup_mask.sum():,}\n")
print("Sample (first 30 rows):")
print(dups.head(30).to_string(index=False))

Rows involved in duplicates: 0

Sample (first 30 rows):
Empty DataFrame
Columns: [ggo, odsek_id, leto_mesec, target, is_a_prediction]
Index: []


In [7]:
# Among duplicated triplets: does 'target' or 'is_a_prediction' vary?
grouped_dups = dups.groupby(ROW_KEY)
print("Columns that vary within a duplicate (odsek_id, ggo, leto_mesec) group:")
for col in ["target", "is_a_prediction"]:
    n_varying = (grouped_dups[col].nunique() > 1).sum()
    print(f"  {col}: varies in {n_varying} groups")

# Check if duplicates are fully identical rows
n_fully_identical = past.duplicated(keep=False).sum()
print(f"\nFully identical rows (all columns same): {n_fully_identical:,}")

Columns that vary within a duplicate (odsek_id, ggo, leto_mesec) group:
  target: varies in 0 groups
  is_a_prediction: varies in 0 groups

Fully identical rows (all columns same): 0


## 3. Segment count comparison: past vs sorted

In [8]:
# How many unique segments and months in each file?
past_segments   = past[SEGMENT_KEY].drop_duplicates().shape[0]
sorted_segments = sorted_df["odsek_id"].nunique()

past_months   = past["leto_mesec"].nunique()
sorted_months = sorted_df["leto_mesec"].nunique()

print(f"{'':30s} {'past':>12} {'sorted':>12}")
print(f"{'Unique segments':30s} {past_segments:>12,} {sorted_segments:>12,}")
print(f"{'Unique months (leto_mesec)':30s} {past_months:>12,} {sorted_months:>12,}")
print(f"{'Total rows':30s} {len(past):>12,} {len(sorted_df):>12,}")
print()
print(f"Expected rows if no dups: {past_segments} segments × {past_months} months = {past_segments * past_months:,}")
print(f"Actual past rows:         {len(past):,}")
print(f"Ratio (actual/expected):  {len(past)/(past_segments * past_months):.2f}x")

                                       past       sorted
Unique segments                      34,417       25,738
Unique months (leto_mesec)              239          240
Total rows                        8,225,663    4,109,455

Expected rows if no dups: 34417 segments × 239 months = 8,225,663
Actual past rows:         8,225,663
Ratio (actual/expected):  1.00x


In [9]:
# Are there odsek_ids in past that map to multiple ggo values?
# (This would explain extra rows vs sorted which only has odsek_id)
ggo_per_odsek = past.groupby("odsek_id")["ggo"].nunique()
print("Distribution of ggo values per odsek_id:")
print(ggo_per_odsek.value_counts().sort_index().rename("num_odsek_ids").to_string())
print()
multi_ggo = ggo_per_odsek[ggo_per_odsek > 1]
print(f"odsek_ids with >1 ggo value: {len(multi_ggo):,}")
if len(multi_ggo) > 0:
    print("\nSample of odsek_ids with multiple ggo:")
    print(past[past["odsek_id"].isin(multi_ggo.head(3).index)][["odsek_id","ggo"]].drop_duplicates().head(12).to_string(index=False))

Distribution of ggo values per odsek_id:
ggo
1    20262
2     3270
3     1340
4      568
5      193
6       48
7       10

odsek_ids with >1 ggo value: 5,429

Sample of odsek_ids with multiple ggo:
odsek_id  ggo
  01001A   10
   01002   10
   01001   11
  01001A   12
   01001    6
   01002    6
   01001    8


## 4. Summary

In [10]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

for name, df in [("past", past), ("future", future)]:
    is_uniform = df["is_a_prediction"].nunique() == 1
    val = df["is_a_prediction"].iloc[0]
    print(f"\nis_a_prediction in {name}: uniform={is_uniform}, value={val}")

print()
dup_count = past.duplicated(subset=ROW_KEY).sum()
full_dup  = past.duplicated().sum()
print(f"heatmap_past_data duplicate (odsek_id, ggo, leto_mesec): {dup_count:,} rows")
print(f"heatmap_past_data fully identical duplicate rows:         {full_dup:,} rows")
print()
print(f"odsek_ids with >1 ggo in past: {len(multi_ggo):,}")
print(f"  → each such odsek_id appears as 2 distinct segments,")
print(f"    which likely explains why past is ~2× the size of sorted")

SUMMARY

is_a_prediction in past: uniform=True, value=False

is_a_prediction in future: uniform=True, value=True

heatmap_past_data duplicate (odsek_id, ggo, leto_mesec): 0 rows
heatmap_past_data fully identical duplicate rows:         0 rows

odsek_ids with >1 ggo in past: 5,429
  → each such odsek_id appears as 2 distinct segments,
    which likely explains why past is ~2× the size of sorted
